# M4.1 · Convolution ve im2col

Görüntü modellerinin çekirdek işlemi. 3×3 bir girdide 2×2'lik çekirdekle convolution'ı elle yapıyor, saf Python'da altı iç içe döngüyle yazıyor, NumPy'da kayan pencere view'u + einsum ile döngüsüz hâle getiriyor, PyTorch'ta `unfold` (im2col) + matmul ile `F.conv2d`'yi yeniden kurup girdi gradyanını `fold` (col2im) ile hesaplıyor ve en sonda dört uygulamayı yarıştırıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m4-1-conv.html](https://m-gonul.github.io/pytorch-lab/m4-1-conv.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m4_1_conv.py
CHECKS = {
 "b": "import numpy as np\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef ref(x, w, b, s, p):\n    x = np.pad(np.asarray(x, float), ((0, 0), (p, p), (p, p))); w = np.asarray(w, float)\n    F, C, KH, KW = w.shape\n    OH, OW = (x.shape[1] - KH) // s + 1, (x.shape[2] - KW) // s + 1\n    return np.array([[[(x[:, i * s:i * s + KH, j * s:j * s + KW] * w[f]).sum() + b[f] for j in range(OW)] for i in range(OH)] for f in range(F)])\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    R.test(\"elle örnek\", lambda: close(need(ns, \"conv2d_loops\")([[[1, 2, 0], [0, 1, 3], [2, 1, 0]]], [[[[1, 0], [-1, 2]]]], [0]), [[[3, 7], [0, 0]]]))\n    R.test(\"pad2d\", lambda: close(need(ns, \"pad2d\")([[[1, 2]]], 1), [[[0, 0, 0, 0], [0, 1, 2, 0], [0, 0, 0, 0]]]))\n    rng = np.random.default_rng(0)\n\n    def rnd():\n        x = rng.normal(size=(2, 7, 6)); w = rng.normal(size=(3, 2, 3, 3)); b = rng.normal(size=3)\n        for s, p in [(1, 0), (2, 1), (3, 2)]:\n            close(need(ns, \"conv2d_loops\")(x.tolist(), w.tolist(), b.tolist(), s, p), ref(x, w, b, s, p), what=f\"stride {s}, pad {p}\")\n    R.test(\"2 kanal → 3 çekirdek, farklı stride/padding\", rnd)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef ref(x, w, b, s, p):\n    x = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)))\n    N, C, H, W = x.shape; F, _, KH, KW = w.shape\n    OH, OW = (H - KH) // s + 1, (W - KW) // s + 1\n    y = np.zeros((N, F, OH, OW))\n    for i in range(OH):\n        for j in range(OW):\n            y[:, :, i, j] = np.tensordot(x[:, :, i * s:i * s + KH, j * s:j * s + KW], w, axes=([1, 2, 3], [1, 2, 3])) + b\n    return y\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    x = rng.normal(size=(2, 3, 9, 8)); w = rng.normal(size=(4, 3, 3, 3)); b = rng.normal(size=4)\n\n    def cv():\n        for s, p in [(1, 0), (1, 1), (2, 1), (2, 2)]:\n            out = need(ns, \"conv2d_np\")(x, w, b, s, p)\n            want = ref(x, w, b, s, p)\n            ensure(out.shape == want.shape, f\"stride {s}, pad {p}: shape {out.shape}, {want.shape} bekleniyordu\")\n            close(out, want, what=f\"stride {s}, pad {p}\")\n    R.test(\"conv2d_np: farklı stride/padding\", cv)\n\n    def ic():\n        for s, p in [(1, 0), (2, 1)]:\n            cols = need(ns, \"im2col_np\")(x, 3, 3, s, p)\n            want = ref(x, w, np.zeros(4), s, p)\n            ensure(cols.shape == (2, 27, want.shape[2] * want.shape[3]), f\"im2col shape {cols.shape}\")\n            close((w.reshape(4, -1) @ cols).reshape(want.shape), want, what=f\"W_mat @ im2col, stride {s}\")\n    R.test(\"im2col_np: W_mat @ cols = conv\", ic)\n",
 "d": "import ast\nimport torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    g = torch.Generator().manual_seed(0)\n    x = torch.randn(2, 3, 9, 8, generator=g, dtype=torch.float64)\n    w = torch.randn(5, 3, 3, 3, generator=g, dtype=torch.float64)\n    b = torch.randn(5, generator=g, dtype=torch.float64)\n\n    def fw():\n        for s, p in [(1, 0), (1, 1), (2, 1)]:\n            close(need(ns, \"conv2d_unfold\")(x, w, b, s, p), F.conv2d(x, w, b, stride=s, padding=p), what=f\"stride {s}, pad {p}\")\n    R.test(\"conv2d_unfold = F.conv2d\", fw)\n\n    def gr():\n        xa, wa, ba = (t.clone().requires_grad_(True) for t in (x, w, b))\n        xb, wb, bb = (t.clone().requires_grad_(True) for t in (x, w, b))\n        dout = torch.randn(2, 5, 5, 4, generator=g, dtype=torch.float64)\n        (need(ns, \"conv2d_unfold\")(xa, wa, ba, 2, 1) * dout).sum().backward()\n        (F.conv2d(xb, wb, bb, stride=2, padding=1) * dout).sum().backward()\n        close(xa.grad, xb.grad, what=\"dx\"); close(wa.grad, wb.grad, what=\"dW\"); close(ba.grad, bb.grad, what=\"db\")\n    R.test(\"autograd gradyanları F.conv2d ile aynı\", gr)\n\n    def ig():\n        fn = next((n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef) and n.name == \"conv2d_input_grad\"), None)\n        ensure(fn is not None, \"`conv2d_input_grad` yok\")\n        names = {n.attr for n in ast.walk(fn) if isinstance(n, ast.Attribute)}\n        ensure(not ({\"backward\", \"grad\", \"conv_transpose2d\"} & names), \"autograd/conv_transpose2d kullanmadan, fold ile yaz\")\n        for s, p in [(1, 0), (2, 1), (1, 2)]:\n            xr = x.clone().requires_grad_(True)\n            out = F.conv2d(xr, w, None, stride=s, padding=p)\n            dout = torch.randn(out.shape, generator=g, dtype=torch.float64)\n            (out * dout).sum().backward()\n            close(need(ns, \"conv2d_input_grad\")(dout, w, x.shape, s, p), xr.grad, what=f\"stride {s}, pad {p}\")\n        return \"dx = fold(W_matᵀ · dY): col2im örtüşen pencereleri topluyor\"\n    R.test(\"conv2d_input_grad (fold) = autograd dx\", ig)\n",
 "e": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef loops(x, w, b):\n    N, C, H, W = x.shape; Fo, _, KH, KW = w.shape\n    y = torch.empty(N, Fo, H - KH + 1, W - KW + 1)\n    for i in range(H - KH + 1):\n        for j in range(W - KW + 1):\n            y[:, :, i, j] = torch.einsum(\"nckl,fckl->nf\", x[:, :, i:i + KH, j:j + KW], w) + b\n    return y\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    x = torch.randn(2, 3, 10, 9, generator=g, dtype=torch.float64); w = torch.randn(4, 3, 3, 3, generator=g, dtype=torch.float64); b = torch.randn(4, generator=g, dtype=torch.float64)\n\n    def eq():\n        for s, p in [(1, 0), (2, 1), (1, 2)]:\n            close(need(ns, \"conv2d_strided\")(x, w, b, s, p), F.conv2d(x, w, b, stride=s, padding=p), what=f\"stride {s}, pad {p}\")\n    R.test(\"conv2d_strided = F.conv2d\", eq)\n    if \"conv2d_strided\" not in ns:\n        return\n    xb = torch.randn(16, 32, 32, 32, generator=g); wb = torch.randn(64, 32, 3, 3, generator=g); bb = torch.randn(64, generator=g)\n\n    def unf():\n        return (wb.reshape(64, -1) @ F.unfold(xb, 3, padding=1) + bb[:, None]).reshape(16, 64, 32, 32)\n    xs, ws = xb[:2, :8, :16, :16], wb[:8, :8]\n    t_loop = bench(lambda: loops(xs, ws, bb[:8]), repeat=3, number=1)\n    t_vs = bench(lambda: ns[\"conv2d_strided\"](xs, ws, bb[:8], 1, 0), repeat=3)\n    t_unf, t_str, t_ref = bench(unf, repeat=3), bench(lambda: ns[\"conv2d_strided\"](xb, wb, bb, 1, 1), repeat=3), bench(lambda: F.conv2d(xb, wb, bb, padding=1), repeat=3)\n    R.table(\"Convolution yolları\", [\"yöntem\", \"girdi\", \"süre\"],\n            [[\"konum döngüsü + einsum\", \"(2, 8, 16, 16)\", fmt_time(t_loop)], [\"senin conv2d_strided\", \"(2, 8, 16, 16)\", fmt_time(t_vs)],\n             [\"F.unfold + matmul (im2col, kopya)\", \"(16, 32, 32, 32)\", fmt_time(t_unf)], [\"senin conv2d_strided (view)\", \"(16, 32, 32, 32)\", fmt_time(t_str)],\n             [\"F.conv2d (oneDNN/cuDNN)\", \"(16, 32, 32, 32)\", fmt_time(t_ref)]],\n            note=\"İm2col matrisi bu örnekte 9 kat bellek: 16·288·1024 eleman. Kütüphane kernel'ları bunu oluşturmadan (implicit GEMM, Winograd) hesaplar.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m4_1_conv as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir convolution katmanı, $F$ adet $C\times K_h\times K_w$ çekirdeği girdi üzerinde kaydırır; her konumda pencere ile çekirdeğin iç çarpımı bir çıktı pikseli verir:

$$
Y_{n,f,i,j}=b_f+\sum_{c=1}^{C}\sum_{a=0}^{K_h-1}\sum_{b=0}^{K_w-1}W_{f,c,a,b}\;X^{\text{pad}}_{n,c,\,iS+a,\,jS+b},
\qquad
H_{\text{out}}=\Big\lfloor\frac{H+2P-K_h}{S}\Big\rfloor+1 .
$$

Doğrudan uygulama altı iç içe döngüdür. im2col bunu tek matris çarpımına çevirir: her pencereyi bir sütuna aç ($C K_hK_w\times H_{\text{out}}W_{\text{out}}$ matris), çekirdekleri satırlara diz ($F\times CK_hK_w$), çarp:

$$
Y_{\text{mat}}=W_{\text{mat}}\cdot\text{im2col}(X)+b .
$$

Geri geçişte $dW=dY_{\text{mat}}\,\text{im2col}(X)^\top$; girdi gradyanı ise $W_{\text{mat}}^\top dY_{\text{mat}}$'yi pencere konumlarına geri dağıtan **col2im** (örtüşen pencerelerde toplanır). PyTorch'ta im2col `F.unfold`, col2im `F.fold`.

:::sezgi Görsel korteksten gelen fikir
Hubel ve Wiesel (1959, 1962) kedinin birincil görsel korteksinde (V1) her nöronun görüş alanının yalnız küçük bir bölgesine (**alıcı alan**) tepki verdiğini, "basit" hücrelerin belirli yönelimli kenarlara, "karmaşık" hücrelerin ise aynı kenara biraz farklı konumlarda da tepki verdiğini buldu. Görme yolunda (V1 → V2 → V4 → IT) alıcı alanlar büyür, öznitelikler karmaşıklaşır. Konvolüsyon ağı bu üç gözlemi mimariye çevirir: **yerel bağlantı** (küçük çekirdek = alıcı alan), **ağırlık paylaşımı** (aynı kenar dedektörü her yerde: öteleme eşdeğerliği), **havuzlama** (karmaşık hücre gibi küçük kaymalara dayanıklılık) ve katmanlar boyunca büyüyen alıcı alan. Fukushima'nın Neocognitron'u (1980) bu fikrin ilk ağıydı; LeCun (1989) gradyanla eğitti. Benzetme esin kaynağıdır, model değil: gerçek korteks geri besleme bağlantıları ve ağırlık paylaşmayan nöronlarla çalışır.
:::

Aşağıdaki deneyde çekirdek, adım, dolgu ve genişlemenin çıktıyı ve boyunu nasıl değiştirdiğini gör.

## 1 · Elle

Çekirdeği girdinin üzerinde kaydır ve her konumda dört çarpımı topla. $Y_{21}$ ve $Y_{22}$'nin 0 çıkması tesadüf değil: çekirdek ($\begin{smallmatrix}1&0\\-1&2\end{smallmatrix}$) bir tür kenar/fark dedektörü.

İm2col'u kâğıtta kur: her 2×2 pencereyi 4 elemanlı bir sütun yap (sol üstten sağ alta), çekirdeği 4 elemanlı bir satır yap. Satır × matris çarpımı dört çıktıyı tek seferde verir; kontrol et.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$X=\begin{bmatrix}1&2&0\\0&1&3\\2&1&0\end{bmatrix}$, $K=\begin{bmatrix}1&0\\-1&2\end{bmatrix}$, stride 1, padding yok (çapraz korelasyon)**

- `y11`: $Y_{11}$
- `y12`: $Y_{12}$
- `y21`: $Y_{21}$
- `y22`: $Y_{22}$

**Boyutlar**

- `o1`: $H=32$, $K=5$, $P=2$, $S=1$: çıktı yüksekliği
- `o2`: $H=32$, $K=3$, $P=1$, $S=2$
- `params`: Conv $3\to64$ kanal, $3\times3$, bias'lı: parametre
- `cols`: Üstteki örnekte im2col matrisinin sütun (pencere) sayısı

In [ ]:
cevaplar = dict(
    y11=None,
    y12=None,
    y21=None,
    y22=None,
    o1=None,
    o2=None,
    params=None,
    cols=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"y11":[3,0],"y12":[7,0],"y21":[0,0],"y22":[0,0],"o1":[32,0],"o2":[16,0],"params":[1792,0],"cols":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, iç içe listeler; altı döngü serbest (CS231n `conv_forward_naive`).

- `pad2d(x, p)` — `x: (C, H, W)` listesi, kenarlara `p` sıfır.
- `conv2d_loops(x, w, b, stride=1, pad=0)` — `x: (C, H, W)`, `w: (F, C, KH, KW)`, `b: (F,)` → `(F, OH, OW)`.

In [ ]:
%%writefile stage_b.py
def pad2d(x, p):
    ...


def conv2d_loops(x, w, b, stride=1, pad=0):
    ...


x = [[[1, 2, 0], [0, 1, 3], [2, 1, 0]]]
w = [[[[1, 0], [-1, 2]]]]
print(conv2d_loops(x, w, [0]))      # [[[3, 7], [0, 0]]]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy'da **döngüsüz**, batch'li:

1. `np.pad(x, ((0, 0), (0, 0), (p, p), (p, p)))`
2. `np.lib.stride_tricks.sliding_window_view(xp, (KH, KW), axis=(2, 3))` → `(N, C, H', W', KH, KW)` kopyasız view; stride için `[:, :, ::S, ::S]`.
3. `np.einsum("nchwkl,fckl->nfhw", windows, w)` + bias.

- `conv2d_np(x, w, b, stride=1, pad=0)` → `(N, F, OH, OW)`.
- `im2col_np(x, KH, KW, stride=1, pad=0)` → `(N, C·KH·KW, OH·OW)`; satır sırası `(c, a, b)`, sütun sırası `(i, j)`. Kontrol, `w.reshape(F, -1) @ im2col` ile conv'un aynı olduğunu doğrular.

In [ ]:
%%writefile stage_c.py
import numpy as np


def conv2d_np(x, w, b, stride=1, pad=0):
    """x: (N, C, H, W), w: (F, C, KH, KW) → (N, F, OH, OW). Döngü yok."""
    ...


def im2col_np(x, KH, KW, stride=1, pad=0):
    """→ (N, C·KH·KW, OH·OW). Döngü yok."""
    ...


x = np.array([[[[1, 2, 0], [0, 1, 3], [2, 1, 0]]]], dtype=float)
w = np.array([[[[1, 0], [-1, 2]]]], dtype=float)
print(conv2d_np(x, w, np.zeros(1)))
print(im2col_np(x, 2, 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch'ta im2col `F.unfold(x, kernel_size, padding=, stride=)` → `(N, C·KH·KW, L)`; col2im `F.fold(cols, output_size, kernel_size, padding=, stride=)`.

- `conv2d_unfold(x, w, b, stride=1, pad=0)` — `unfold` + matmul + reshape; `F.conv2d` ile aynı çıktı **ve** aynı autograd gradyanları.
- `conv2d_input_grad(dout, w, x_shape, stride=1, pad=0)` → `dx`; autograd kullanmadan: `cols = W_matᵀ @ dout_mat`, sonra `F.fold`. Kontrol, autograd'ın verdiği `dx` ile karşılaştırır.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def conv2d_unfold(x, w, b, stride=1, pad=0):
    ...


def conv2d_input_grad(dout, w, x_shape, stride=1, pad=0):
    """dx, autograd kullanmadan (F.fold ile)."""
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`unfold`'un bir de tensör metodu var ve **kopyasız** view üretir: `x.unfold(2, KH, S).unfold(3, KW, S)` → `(N, C, OH, OW, KH, KW)`. Bunu einsum'a vermek, `F.unfold`'un kopyalayan im2col'una göre bellek tasarrufu sağlar.

- `conv2d_strided(x, w, b, stride=1, pad=0)` — `F.pad` + iki `.unfold` + `torch.einsum`. **Döngü yok.**

Kontrol doğruluğu `F.conv2d` ile karşılaştırır, sonra dört yolu yarıştırır: Python döngüleri (küçük girdide), `F.unfold` + matmul, strided einsum ve `F.conv2d`.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F


def conv2d_strided(x, w, b, stride=1, pad=0):
    """F.pad + iki .unfold (view) + einsum. Döngü yok."""
    ...

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Mathieu, Henaff, LeCun (2014): FFT ile hızlı CNN eğitimi

[Mathieu, Henaff, LeCun — Fast Training of Convolutional Networks through FFTs (ICLR 2014)](https://arxiv.org/abs/1312.5851)

Konvolüsyonu FFT ile hesapla: girdiyi ve çekirdeği sıfır dolgulayıp frekans uzayında çarp, ters dönüşüm al; çapraz korelasyon için çekirdeği çevir. Doğrudan, im2col ve FFT yöntemlerini çekirdek boyutu k ∈ {3, 7, 15, 31} ve görüntü boyutuna göre karşılaştır; makalenin maliyet analizini (Bölüm 3: FFT'lerin girdi/çekirdek/çıktı için yeniden kullanımı) ileri ve geri geçişin üç konvolüsyonu üzerinde uygula. Kesişim noktasını M10.7'deki sonuçla karşılaştır.


**Adımlar**

1. Dairesel konvolüsyonun 'valid' kısmını doğru kırp.

**Kabul ölçütleri**

- Üç yöntem aynı sonucu verir.
- Büyük k'da FFT'nin üstün olduğu kesişim noktası.

## Kaynaklar

- [CS231n — Convolutional Neural Networks (conv layer, im2col)](https://cs231n.github.io/convolutional-networks/) — MIT
- [CS231n Assignment 2 — conv_forward_naive / conv_backward_naive](https://cs231n.github.io/assignments2025/assignment2/) — MIT
- [Dive into Deep Learning §7.2–7.3 Convolutions for Images, Padding and Stride](https://d2l.ai/chapter_convolutional-neural-networks/index.html) — CC BY-SA 4.0 / MIT-0
- [Dumoulin & Visin — A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285) — arXiv